# কল গ্রাফ বিশ্লেষণ এবং স্ট্রংলি কানেক্টেড কম্পোনেন্টস (SCC) সনাক্তকরণ

এই নোটবুকটি UnifyWeaver-এর উন্নত কোড বিশ্লেষণ ক্ষমতাগুলো অন্বেষণ করে:

- **কল গ্রাফ গঠন** — Prolog কোড থেকে নির্ভরতা গ্রাফ তৈরি
- **SCC সনাক্তকরণ** — দৃঢ়ভাবে সংযুক্ত উপাদান খুঁজে বের করা (মিউচুয়াল রিকার্শন)
- **প্যাটার্ন বিশ্লেষণ** — রিকার্শন প্যাটার্ন বোঝা
- **নির্ভরতা ভিজ্যুয়ালাইজেশন** — প্রেডিকেটের সম্পর্কের দৃশ্যরূপ

## শেখার উদ্দেশ্য

- UnifyWeaver কীভাবে কোড কাঠামো বিশ্লেষণ করে তা বোঝা
- কল গ্রাফ তৈরি ও পরীক্ষা করা
- টারজান (Tarjan) অ্যালগরিদম ব্যবহার করে মিউচুয়াল রিকার্শন সনাক্ত করা
- কোড নির্ভরতা ভিজ্যুয়ালাইজ করা

## সেটআপ

UnifyWeaver এবং বিশ্লেষণ মডিউল লোড করুন।

In [ ]:
% Load initialization
['../init'].

% Load analysis modules
use_module(unifyweaver(core/advanced/call_graph)).
use_module(unifyweaver(core/advanced/scc_detection)).
use_module(unifyweaver(core/advanced/pattern_matchers)).

## উদাহরণ ১: সহজ কল গ্রাফ

একটি সাধারণ প্রেডিকেট দিয়ে শুরু করে তার কল গ্রাফ তৈরি করি।

In [ ]:
% Define ancestor predicate
:- dynamic ancestor/2.
:- dynamic parent/2.

% Parent facts
parent(abraham, isaac).
parent(isaac, jacob).

% Ancestor rules
ancestor(X, Y) :- parent(X, Y).
ancestor(X, Z) :- parent(X, Y), ancestor(Y, Z).

### কল গ্রাফ তৈরি করুন

In [ ]:
% Build call graph for ancestor
build_call_graph([ancestor/2], _Graph),
format('Call Graph for ancestor/2:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### নির্ভরতা বিশ্লেষণ করুন

In [ ]:
% Get all dependencies of ancestor/2
get_dependencies(ancestor/2, _Deps),
format('Dependencies of ancestor/2: ~w~n', [_Deps]).

% Check if self-recursive
(is_self_recursive(ancestor/2) ->
    writeln('✓ ancestor/2 is self-recursive')
;
    writeln('✗ ancestor/2 is not self-recursive')
).

## উদাহরণ ২: মিউচুয়াল রিকার্শন সনাক্তকরণ

এখন জোড়/বিজোড় উদাহরণ দিয়ে মিউচুয়াল রিকার্শন সনাক্ত করি।

In [ ]:
% Define mutually recursive predicates
:- dynamic is_even/1.
:- dynamic is_odd/1.

is_even(0).
is_even(N) :- N > 0, N1 is N - 1, is_odd(N1).

is_odd(1).
is_odd(N) :- N > 1, N1 is N - 1, is_even(N1).

### উভয় প্রেডিকেটের জন্য কল গ্রাফ তৈরি করুন

In [ ]:
% Build call graph for both predicates
build_call_graph([is_even/1, is_odd/1], _Graph),
format('Call Graph for is_even/1 and is_odd/1:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### স্ট্রংলি কানেক্টেড কম্পোনেন্টস (SCC) খুঁজুন

In [ ]:
% Rebuild the graph because variables do not persist between notebook cells
build_call_graph([is_even/1, is_odd/1], _Graph),
% Find SCCs using Tarjan's algorithm
find_sccs(_Graph, _SCCs),
format('Strongly Connected Components:~n'),
forall(member(_SCC, _SCCs),
    format('  ~w~n', [_SCC])).

### SCC তুচ্ছ (Trivial) কিনা তা পরীক্ষা করুন

In [ ]:
% Rebuild the derived values so this cell also runs independently
build_call_graph([is_even/1, is_odd/1], _Graph),
find_sccs(_Graph, _SCCs),
% Check each SCC
forall(member(_SCC, _SCCs),
    (   is_trivial_scc(_SCC) ->
        format('  ~w is trivial (single predicate)~n', [_SCC])
    ;
        format('  ~w is NON-TRIVIAL (mutual recursion!)~n', [_SCC])
    )
).

## উদাহরণ ৩: জটিল কল গ্রাফ

একাধিক প্রেডিকেট সহ একটি জটিল সিস্টেম বিশ্লেষণ করি।

In [ ]:
% Define a small program with multiple predicates
:- dynamic grandparent/2.
:- dynamic sibling/2.
:- dynamic cousin/2.

% grandparent uses parent
grandparent(X, Z) :- parent(X, Y), parent(Y, Z).

% sibling: same parent, different children
sibling(X, Y) :- parent(P, X), parent(P, Y), X \= Y.

% cousin: parents are siblings
cousin(X, Y) :- parent(P1, X), parent(P2, Y), sibling(P1, P2).

### সম্পূর্ণ কল গ্রাফ তৈরি করুন

In [ ]:
% Build call graph for all predicates
build_call_graph([grandparent/2, sibling/2, cousin/2], _Graph),
format('Complete Call Graph:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### প্রেডিকেট গ্রুপ খুঁজুন

একটি শুরুর প্রেডিকেট ধারণকারী পারস্পরিক রিকার্সিভ প্রেডিকেট গ্রুপটি সন্ধান করুন।

In [ ]:
% Find the mutually recursive group containing cousin/2
predicates_in_group(cousin/2, _Group),
format('Mutually recursive group containing cousin/2: ~w~n', [_Group]).

## উদাহরণ ৪: প্যাটার্ন সনাক্তকরণ

রিকার্শনের ধরন বিশ্লেষণ করতে প্যাটার্ন ম্যাচার ব্যবহার করুন।

In [ ]:
% Define various recursive patterns
:- dynamic count/3.     % Tail recursive
:- dynamic factorial/2. % Linear recursive
:- dynamic fib/2.       % Tree recursive (or linear if detected)

% Tail recursive count
count([], Acc, Acc).
count([_|T], Acc, N) :-
    Acc1 is Acc + 1,
    count(T, Acc1, N).

% Linear recursive factorial
factorial(0, 1).
factorial(N, F) :-
    N > 0,
    N1 is N - 1,
    factorial(N1, F1),
    F is N * F1.

% Fibonacci (can be detected as linear or tree)
fib(0, 0).
fib(1, 1).
fib(N, F) :-
    N > 1,
    N1 is N - 1,
    N2 is N - 2,
    fib(N1, F1),
    fib(N2, F2),
    F is F1 + F2.

### টেইল রিকার্শন সনাক্ত করুন

In [ ]:
% Check if count/3 is tail recursive
(is_tail_recursive_accumulator(count/3, _AccInfo) ->
    format('✓ count/3 is tail recursive: ~w~n', [_AccInfo])
;
    writeln('✗ count/3 is not tail recursive')
).

### লিনিয়ার রিকার্শন সনাক্ত করুন

In [ ]:
% Check if factorial/2 is linear recursive
(is_linear_recursive_streamable(factorial/2) ->
    writeln('✓ factorial/2 is linear recursive')
;
    writeln('✗ factorial/2 is not linear recursive')
).

### রিকার্সিভ কলের সংখ্যা গণনা করুন

In [ ]:
% Count recursive calls in fibonacci
functor(_FibHead, fib, 2),
user:clause(_FibHead, _FibBody),
once(contains_call_to(_FibBody, fib)),
count_recursive_calls(_FibBody, fib, _Count),
format('Fibonacci body has ~w recursive calls~n', [_Count]).

## DOT ফরম্যাটের মাধ্যমে ভিজ্যুয়ালাইজেশন

আমাদের কল গ্রাফের একটি Graphviz DOT উপস্থাপন তৈরি করি।

In [ ]:
% Helper to generate DOT format
generate_dot(Graph, DotCode) :-
    findall(Line,
        (   member(From -> To, Graph),
            format(atom(Line), '  "~w" -> "~w";', [From, To])
        ),
        Lines),
    atomic_list_concat(['digraph CallGraph {', '  rankdir=LR;' | Lines], '\n', Body),
    format(atom(DotCode), '~w~n}~n', [Body]).

% Generate DOT for even/odd graph
build_call_graph([is_even/1, is_odd/1], _Graph),
generate_dot(_Graph, _DotCode),
writeln('DOT Code for even/odd call graph:'),
writeln(_DotCode).

### DOT ফাইল সংরক্ষণ করুন

In [ ]:
% Rebuild the DOT source because variables do not persist between cells
build_call_graph([is_even/1, is_odd/1], _Graph),
generate_dot(_Graph, _DotCode),
setup_call_cleanup(
    open('/shared/data/even_odd_graph.dot', write, _Stream),
    write(_Stream, _DotCode),
    close(_Stream)),
writeln('✓ Saved to /shared/data/even_odd_graph.dot').

writeln('To visualize, run:'),
writeln('  dot -Tpng /shared/data/even_odd_graph.dot -o even_odd_graph.png').

## অনুশীলন: আপনার নিজস্ব কোড বিশ্লেষণ করুন

আপনার নিজস্ব প্রেডিকেট সংজ্ঞায়িত এবং বিশ্লেষণ করার চেষ্টা করুন!

In [ ]:
% Define your predicates here
% Then build call graphs, find SCCs, and detect patterns

% Example:
% :- dynamic my_predicate/2.
% my_predicate(...) :- ...

% build_call_graph([my_predicate/2], Graph).


## সারসংক্ষেপ

এই নোটবুকে আপনি শিখেছেন:

✅ কীভাবে Prolog কোড থেকে কল গ্রাফ তৈরি করতে হয়

✅ মিউচুয়াল রিকার্শনের জন্য কীভাবে স্ট্রংলি কানেক্টেড কম্পোনেন্টস (SCC) সনাক্ত করতে হয়

✅ রিকার্শনের ধরন শ্রেণীবদ্ধ করতে কীভাবে প্যাটার্ন ম্যাচার ব্যবহার করতে হয়

✅ কীভাবে প্রেডিকেটের নির্ভরতা বিশ্লেষণ করতে হয়

✅ DOT ফরম্যাট ব্যবহার করে কীভাবে কল গ্রাফ ভিজ্যুয়ালাইজ করতে হয়

## উন্নত বিষয়সমূহ

আরও গভীর বিশ্লেষণের জন্য:

- **টপোলজিক্যাল সর্টিং**: নির্ভরতার ভিত্তিতে SCC সাজাতে `topological_order/2` ব্যবহার করুন
- **কাস্টম প্যাটার্ন ম্যাচার**: নিজস্ব প্যাটার্ন সনাক্তকরণ প্রেডিকেট লিখুন
- **অ্যাকুমুলেটর প্যাটার্ন নিষ্কাশন**: বিস্তারিত বিশ্লেষণের জন্য `extract_accumulator_pattern/2` ব্যবহার করুন
- **লিনিয়ার রিকার্শন নিষিদ্ধ করা**: ভিন্ন সংকলন কৌশল প্রয়োগের জন্য `forbid_linear_recursion/1` ব্যবহার করুন

## রেফারেন্স এবং সম্পর্কিত ফাইল

- অধ্যায় ১০: Prolog ইন্ট্রোস্পেকশন এবং তত্ত্ব
- `src/unifyweaver/core/advanced/call_graph.pl`
- `src/unifyweaver/core/advanced/scc_detection.pl`
- `src/unifyweaver/core/advanced/pattern_matchers.pl`